# Classical Approaches


## Import Libraries

In [1]:
import pandas as pd
import importlib 
import os
import sys

module_path = os.path.abspath(os.path.join(".."))
if module_path not in sys.path:
	sys.path.append(module_path)

import src.utils
importlib.reload(src.utils)

from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
import xgboost as xgb
import lightgbm as lgb

from pprint import pprint

from src.utils import create_one_hot_encoding, get_unique_champs_from_df

## Bag of Champions + Logistic Regression

### Splitting the Dataset


In [2]:
matches = pd.read_csv("../data/TeamMatchTbl.csv")

dropped_cols = [
	"TeamID", "MatchFk", "BlueBaronKills",
	"BlueRiftHeraldKills", "BlueDragonKills",
	"BlueTowerKills", "BlueKills", "RedBaronKills",
	"RedRiftHeraldKills", "RedDragonKills", "RedTowerKills",
	"RedKills", "RedWin"
]

print("Dropped Columns")
pprint(dropped_cols)

train_val_df, test_df = train_test_split(matches, test_size=0.2, random_state=42)
train_df, val_df = train_test_split(train_val_df, test_size=0.2, random_state=42)

train_df = train_df.drop(columns=dropped_cols)
val_df = val_df.drop(columns=dropped_cols)
test_df = test_df.drop(columns=dropped_cols)

train_df.head()

Dropped Columns
['TeamID',
 'MatchFk',
 'BlueBaronKills',
 'BlueRiftHeraldKills',
 'BlueDragonKills',
 'BlueTowerKills',
 'BlueKills',
 'RedBaronKills',
 'RedRiftHeraldKills',
 'RedDragonKills',
 'RedTowerKills',
 'RedKills',
 'RedWin']


,B1Champ,B2Champ,B3Champ,B4Champ,B5Champ,R1Champ,R2Champ,R3Champ,R4Champ,R5Champ,BlueWin
117776,77,3,887,36,233,106,84,202,85,429,1
105311,83,141,103,901,201,777,28,25,81,518,0
41139,82,234,28,804,26,887,64,136,222,89,0
24402,24,11,13,115,43,36,517,777,202,267,0
118127,904,83,90,901,25,17,266,127,21,350,0


In [3]:
print(train_df.shape)
print(val_df.shape)
print(test_df.shape)

(86517, 11)
(21630, 11)
(27037, 11)


### One-Hot Encoding

In [4]:
blue_champ_cols = [
	"B1Champ", 
	"B2Champ",
	"B3Champ",
	"B4Champ",
	"B5Champ",
]
red_champ_cols = [
	"R1Champ",
	"R2Champ",
	"R3Champ",
	"R4Champ",
	"R5Champ",
]

champ_cols = blue_champ_cols + red_champ_cols
target_cols = ["BlueWin"]

train_unique_champs = get_unique_champs_from_df(train_df, champ_cols)
val_unique_champs = get_unique_champs_from_df(val_df, champ_cols)
test_unique_champs = get_unique_champs_from_df(test_df, champ_cols)

ohe_train = create_one_hot_encoding(
	train_df, 
	blue_champ_cols, 
	red_champ_cols, 
	train_unique_champs, 
	names=True
)

ohe_val = create_one_hot_encoding(
	val_df,
	blue_champ_cols, 
	red_champ_cols, 
	train_unique_champs, 
	names=True
)

ohe_test = create_one_hot_encoding(
	test_df,
	blue_champ_cols,
	red_champ_cols,
	train_unique_champs,
	names=True
)

print(f"Training dataset shape: {ohe_train.shape}")
print("There should be 172 champion cols, 1 target col, and 1 value for each champion (1 for blue team -1 for red")
print(f"Number of cols: {len(ohe_train.columns)}")
print(f"Number of unique cols: {len(ohe_train.columns.unique())}")
print(ohe_train.columns.value_counts())
ohe_train

Training dataset shape: (86517, 173)
There should be 172 champion cols, 1 target col, and 1 value for each champion (1 for blue team -1 for red
Number of cols: 173
Number of unique cols: 173
Annie          1
Olaf           1
Galio          1
TwistedFate    1
Sylas          1
              ..
Bard           1
Naafiri        1
Rakan          1
Xayah          1
BlueWin        1
Name: count, Length: 173, dtype: int64


,Annie,Olaf,Galio,TwistedFate,Sylas,Neeko,Leblanc,XinZhao,Fiddlesticks,Kayle,...,Thresh,Illaoi,RekSai,Ivern,Kalista,Bard,Naafiri,Rakan,Xayah,BlueWin
117776,0,0,1,0,0,0,0,0,0,0,...,0,0,0,0,-1,0,0,0,0,1
105311,0,0,0,0,0,-1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
41139,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
24402,0,0,0,0,-1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
118127,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
107132,0,0,0,0,-1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
67393,0,1,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
123552,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
59363,-1,0,0,0,0,0,0,0,0,-1,...,0,0,0,0,0,0,0,0,0,1


### Training

In [5]:
X_train = ohe_train.drop(columns=target_cols)
y_train = ohe_train[target_cols].values.ravel()

X_val = ohe_val.drop(columns=target_cols)
y_val = ohe_val[target_cols].values.ravel()

X_test = ohe_test.drop(columns=target_cols)
y_test = ohe_test[target_cols].values.ravel()

In [6]:
# Define hyperparameter grid for tuning
param_grid = {
    "C": [0.001, 0.01, 0.1, 1, 10, 100],
}

best_score = 0
best_params = None
best_model = None

for C in param_grid["C"]:
		model = LogisticRegression(
			C=C,
			max_iter=1000,
			random_state=42,
		)

		# Fit on the training data
		model.fit(X_train, y_train)

		# Evaluate on the validation data. We slice to get the probabilities for the positive class.
		val_prob = model.predict_proba(X_val)[:, 1]
		val_score = roc_auc_score(y_val, val_prob)

		print(f"C={C}, Val AUC={val_score:.4f}")

		if val_score > best_score:
			best_score = val_score
			best_model = model
			best_params = { "C": C }


print(f"\nBest parameters: {best_params}")
print(f"Best validation score: {best_score:.4f}")

# Final evaluation on test set
test_proba = best_model.predict_proba(X_test)[:, 1]
test_score = roc_auc_score(y_test, test_proba)
print(f"Final test score: {test_score:.4f}")

C=0.001, Val AUC=0.5301
C=0.01, Val AUC=0.5309
C=0.1, Val AUC=0.5309
C=1, Val AUC=0.5308
C=10, Val AUC=0.5308
C=100, Val AUC=0.5308

Best parameters: {'C': 0.01}
Best validation score: 0.5309
Final test score: 0.5294


## Tree-Based Models

### Prepare Integer-Encoded Features for Tree Models

In [7]:
# For tree-based models, use the original integer encoding (no one-hot encoding needed)
# Trees can naturally handle categorical variables as integers

X_train_tree = train_df[champ_cols]
y_train_tree = train_df["BlueWin"].values

X_val_tree = val_df[champ_cols]
y_val_tree = val_df["BlueWin"].values

X_test_tree = test_df[champ_cols]
y_test_tree = test_df["BlueWin"].values

print(f"Tree-based model feature shape: {X_train_tree.shape}")
print(f"Features: {list(X_train_tree.columns)}")
print(f"\nFirst few rows:")
print(X_train_tree.head())

Tree-based model feature shape: (86517, 10)
Features: ['B1Champ', 'B2Champ', 'B3Champ', 'B4Champ', 'B5Champ', 'R1Champ', 'R2Champ', 'R3Champ', 'R4Champ', 'R5Champ']

First few rows:
        B1Champ  B2Champ  B3Champ  B4Champ  B5Champ  R1Champ  R2Champ  \
117776       77        3      887       36      233      106       84   
105311       83      141      103      901      201      777       28   
41139        82      234       28      804       26      887       64   
24402        24       11       13      115       43       36      517   
118127      904       83       90      901       25       17      266   

        R3Champ  R4Champ  R5Champ  
117776      202       85      429  
105311       25       81      518  
41139       136      222       89  
24402       777      202      267  
118127      127       21      350  


### Random Forest

In [8]:
# Define hyperparameter grid for Random Forest
param_grid_rf = {
    "n_estimators": [50, 100, 200],
    "max_depth": [10, 20, 30, None],
    "min_samples_split": [2, 5, 10],
}

best_score_rf = 0
best_params_rf = None
best_model_rf = None

print("Random Forest Hyperparameter Tuning (using integer encoding)")
print("=" * 50)

for n_est in param_grid_rf["n_estimators"]:
    for max_d in param_grid_rf["max_depth"]:
        for min_split in param_grid_rf["min_samples_split"]:
            model = RandomForestClassifier(
                n_estimators=n_est,
                max_depth=max_d,
                min_samples_split=min_split,
                random_state=42,
                n_jobs=-1
            )
            
            # Fit on training data using integer-encoded features
            model.fit(X_train_tree, y_train_tree)
            
            # Evaluate on validation data
            val_prob = model.predict_proba(X_val_tree)[:, 1]
            val_score = roc_auc_score(y_val_tree, val_prob)
            
            print(f"n_estimators={n_est}, max_depth={max_d}, min_samples_split={min_split}, Val AUC={val_score:.4f}")
            
            if val_score > best_score_rf:
                best_score_rf = val_score
                best_model_rf = model
                best_params_rf = {
                    "n_estimators": n_est,
                    "max_depth": max_d,
                    "min_samples_split": min_split
                }

print("\n" + "=" * 50)
print(f"Best parameters: {best_params_rf}")
print(f"Best validation score: {best_score_rf:.4f}")

# Final evaluation on test set
test_proba_rf = best_model_rf.predict_proba(X_test_tree)[:, 1]
test_score_rf = roc_auc_score(y_test_tree, test_proba_rf)
print(f"Final test score: {test_score_rf:.4f}")

Random Forest Hyperparameter Tuning (using integer encoding)
n_estimators=50, max_depth=10, min_samples_split=2, Val AUC=0.5696
n_estimators=50, max_depth=10, min_samples_split=5, Val AUC=0.5703
n_estimators=50, max_depth=10, min_samples_split=10, Val AUC=0.5696
n_estimators=50, max_depth=20, min_samples_split=2, Val AUC=0.5586
n_estimators=50, max_depth=20, min_samples_split=5, Val AUC=0.5628
n_estimators=50, max_depth=20, min_samples_split=10, Val AUC=0.5693
n_estimators=50, max_depth=30, min_samples_split=2, Val AUC=0.5580
n_estimators=50, max_depth=30, min_samples_split=5, Val AUC=0.5536
n_estimators=50, max_depth=30, min_samples_split=10, Val AUC=0.5620
n_estimators=50, max_depth=None, min_samples_split=2, Val AUC=0.5547
n_estimators=50, max_depth=None, min_samples_split=5, Val AUC=0.5551
n_estimators=50, max_depth=None, min_samples_split=10, Val AUC=0.5617
n_estimators=100, max_depth=10, min_samples_split=2, Val AUC=0.5716
n_estimators=100, max_depth=10, min_samples_split=5, Val 

### XGBoost

In [9]:
# Define hyperparameter grid for XGBoost
param_grid_xgb = {
    "n_estimators": [50, 100, 200],
    "learning_rate": [0.01, 0.1, 0.3],
    "max_depth": [3, 5, 7, 10],
    "subsample": [0.8, 1.0],
}

best_score_xgb = 0
best_params_xgb = None
best_model_xgb = None

print("XGBoost Hyperparameter Tuning (using integer encoding)")
print("=" * 50)

for n_est in param_grid_xgb["n_estimators"]:
    for lr in param_grid_xgb["learning_rate"]:
        for max_d in param_grid_xgb["max_depth"]:
            for subsample in param_grid_xgb["subsample"]:
                model = xgb.XGBClassifier(
                    n_estimators=n_est,
                    learning_rate=lr,
                    max_depth=max_d,
                    subsample=subsample,
                    random_state=42,
                    n_jobs=-1,
                    eval_metric='logloss'
                )
                
                # Fit on training data using integer-encoded features
                model.fit(X_train_tree, y_train_tree)
                
                # Evaluate on validation data
                val_prob = model.predict_proba(X_val_tree)[:, 1]
                val_score = roc_auc_score(y_val_tree, val_prob)
                
                print(f"n_est={n_est}, lr={lr}, max_d={max_d}, subsample={subsample}, Val AUC={val_score:.4f}")
                
                if val_score > best_score_xgb:
                    best_score_xgb = val_score
                    best_model_xgb = model
                    best_params_xgb = {
                        "n_estimators": n_est,
                        "learning_rate": lr,
                        "max_depth": max_d,
                        "subsample": subsample
                    }

print("\n" + "=" * 50)
print(f"Best parameters: {best_params_xgb}")
print(f"Best validation score: {best_score_xgb:.4f}")

# Final evaluation on test set
test_proba_xgb = best_model_xgb.predict_proba(X_test_tree)[:, 1]
test_score_xgb = roc_auc_score(y_test_tree, test_proba_xgb)
print(f"Final test score: {test_score_xgb:.4f}")

XGBoost Hyperparameter Tuning (using integer encoding)
n_est=50, lr=0.01, max_d=3, subsample=0.8, Val AUC=0.5512
n_est=50, lr=0.01, max_d=3, subsample=1.0, Val AUC=0.5458
n_est=50, lr=0.01, max_d=5, subsample=0.8, Val AUC=0.5622
n_est=50, lr=0.01, max_d=5, subsample=1.0, Val AUC=0.5585
n_est=50, lr=0.01, max_d=7, subsample=0.8, Val AUC=0.5696
n_est=50, lr=0.01, max_d=7, subsample=1.0, Val AUC=0.5677
n_est=50, lr=0.01, max_d=10, subsample=0.8, Val AUC=0.5765
n_est=50, lr=0.01, max_d=10, subsample=1.0, Val AUC=0.5705
n_est=50, lr=0.1, max_d=3, subsample=0.8, Val AUC=0.5801
n_est=50, lr=0.1, max_d=3, subsample=1.0, Val AUC=0.5810
n_est=50, lr=0.1, max_d=5, subsample=0.8, Val AUC=0.5881
n_est=50, lr=0.1, max_d=5, subsample=1.0, Val AUC=0.5885
n_est=50, lr=0.1, max_d=7, subsample=0.8, Val AUC=0.5910
n_est=50, lr=0.1, max_d=7, subsample=1.0, Val AUC=0.5906
n_est=50, lr=0.1, max_d=10, subsample=0.8, Val AUC=0.5868
n_est=50, lr=0.1, max_d=10, subsample=1.0, Val AUC=0.5916
n_est=50, lr=0.3, max

### LightGBM

In [10]:
# Define hyperparameter grid for LightGBM
param_grid_lgb = {
    "n_estimators": [50, 100, 200],
    "learning_rate": [0.01, 0.1, 0.3],
    "max_depth": [3, 5, 7, 10],
    "num_leaves": [31, 63, 127],
}

best_score_lgb = 0
best_params_lgb = None
best_model_lgb = None

print("LightGBM Hyperparameter Tuning (using integer encoding)")
print("=" * 50)

for n_est in param_grid_lgb["n_estimators"]:
    for lr in param_grid_lgb["learning_rate"]:
        for max_d in param_grid_lgb["max_depth"]:
            for n_leaves in param_grid_lgb["num_leaves"]:
                model = lgb.LGBMClassifier(
                    n_estimators=n_est,
                    learning_rate=lr,
                    max_depth=max_d,
                    num_leaves=n_leaves,
                    random_state=42,
                    n_jobs=-1,
                    verbose=-1
                )
                
                # Fit on training data using integer-encoded features
                model.fit(X_train_tree, y_train_tree)
                
                # Evaluate on validation data
                val_prob = model.predict_proba(X_val_tree)[:, 1]
                val_score = roc_auc_score(y_val_tree, val_prob)
                
                print(f"n_est={n_est}, lr={lr}, max_d={max_d}, n_leaves={n_leaves}, Val AUC={val_score:.4f}")
                
                if val_score > best_score_lgb:
                    best_score_lgb = val_score
                    best_model_lgb = model
                    best_params_lgb = {
                        "n_estimators": n_est,
                        "learning_rate": lr,
                        "max_depth": max_d,
                        "num_leaves": n_leaves
                    }

print("\n" + "=" * 50)
print(f"Best parameters: {best_params_lgb}")
print(f"Best validation score: {best_score_lgb:.4f}")

# Final evaluation on test set
test_proba_lgb = best_model_lgb.predict_proba(X_test_tree)[:, 1]
test_score_lgb = roc_auc_score(y_test_tree, test_proba_lgb)
print(f"Final test score: {test_score_lgb:.4f}")

LightGBM Hyperparameter Tuning (using integer encoding)
n_est=50, lr=0.01, max_d=3, n_leaves=31, Val AUC=0.5458
n_est=50, lr=0.01, max_d=3, n_leaves=63, Val AUC=0.5458
n_est=50, lr=0.01, max_d=3, n_leaves=127, Val AUC=0.5458
n_est=50, lr=0.01, max_d=5, n_leaves=31, Val AUC=0.5587
n_est=50, lr=0.01, max_d=5, n_leaves=63, Val AUC=0.5587
n_est=50, lr=0.01, max_d=5, n_leaves=127, Val AUC=0.5587
n_est=50, lr=0.01, max_d=7, n_leaves=31, Val AUC=0.5695
n_est=50, lr=0.01, max_d=7, n_leaves=63, Val AUC=0.5686
n_est=50, lr=0.01, max_d=7, n_leaves=127, Val AUC=0.5685
n_est=50, lr=0.01, max_d=10, n_leaves=31, Val AUC=0.5758
n_est=50, lr=0.01, max_d=10, n_leaves=63, Val AUC=0.5740
n_est=50, lr=0.01, max_d=10, n_leaves=127, Val AUC=0.5733
n_est=50, lr=0.1, max_d=3, n_leaves=31, Val AUC=0.5805
n_est=50, lr=0.1, max_d=3, n_leaves=63, Val AUC=0.5805
n_est=50, lr=0.1, max_d=3, n_leaves=127, Val AUC=0.5805
n_est=50, lr=0.1, max_d=5, n_leaves=31, Val AUC=0.5890
n_est=50, lr=0.1, max_d=5, n_leaves=63, Val 

# Model Comparison

In [11]:
# Compare all models
comparison_df = pd.DataFrame({
    'Model': ['Logistic Regression', 'Random Forest', 'XGBoost', 'LightGBM'],
    'Validation AUC': [best_score, best_score_rf, best_score_xgb, best_score_lgb],
    'Test AUC': [test_score, test_score_rf, test_score_xgb, test_score_lgb],
    'Encoding': ['One-Hot (Diff)', 'Integer', 'Integer', 'Integer'],
    'Best Parameters': [best_params, best_params_rf, best_params_xgb, best_params_lgb]
})

print("Model Performance Comparison")
print("=" * 100)
print(comparison_df[['Model', 'Validation AUC', 'Test AUC', 'Encoding']].to_string(index=False))
print("\n" + "=" * 100)

# Find the best model
best_overall_idx = comparison_df['Test AUC'].idxmax()
best_overall_model = comparison_df.iloc[best_overall_idx]['Model']
best_overall_score = comparison_df.iloc[best_overall_idx]['Test AUC']

print(f"\nBest Overall Model: {best_overall_model}")
print(f"Test AUC: {best_overall_score:.4f}")

print("\n" + "=" * 100)
print("\nDetailed Parameters:")
for idx, row in comparison_df.iterrows():
    print(f"\n{row['Model']}:")
    print(f"  {row['Best Parameters']}")

Model Performance Comparison
              Model  Validation AUC  Test AUC       Encoding
Logistic Regression        0.530855  0.529440 One-Hot (Diff)
      Random Forest        0.576442  0.581504        Integer
            XGBoost        0.600376  0.599346        Integer
           LightGBM        0.600125  0.594806        Integer


Best Overall Model: XGBoost
Test AUC: 0.5993


Detailed Parameters:

Logistic Regression:
  {'C': 0.01}

Random Forest:
  {'n_estimators': 200, 'max_depth': 20, 'min_samples_split': 10}

XGBoost:
  {'n_estimators': 200, 'learning_rate': 0.3, 'max_depth': 3, 'subsample': 1.0}

LightGBM:
  {'n_estimators': 200, 'learning_rate': 0.1, 'max_depth': 10, 'num_leaves': 31}


# Save Best Model

In [12]:
import pickle
import json
from pathlib import Path

# Create models directory if it doesn't exist
models_dir = Path("../models")
models_dir.mkdir(exist_ok=True)

# Find best tree-based model (exclude Logistic Regression)
tree_models = comparison_df[comparison_df['Encoding'] == 'Integer'].copy()
best_tree_idx = tree_models['Test AUC'].idxmax()
best_tree_model_name = tree_models.loc[best_tree_idx, 'Model']
best_tree_score = tree_models.loc[best_tree_idx, 'Test AUC']
best_tree_params = tree_models.loc[best_tree_idx, 'Best Parameters']

# Get the actual model object
model_map = {
    'Random Forest': best_model_rf,
    'XGBoost': best_model_xgb,
    'LightGBM': best_model_lgb
}

best_tree_model = model_map[best_tree_model_name]

# Save the model
model_path = models_dir / "best_tree_model.pkl"
with open(model_path, 'wb') as f:
    pickle.dump(best_tree_model, f)

# Save metadata
metadata = {
    'model_type': best_tree_model_name,
    'test_auc': float(best_tree_score),
    'validation_auc': float(tree_models.loc[best_tree_idx, 'Validation AUC']),
    'hyperparameters': best_tree_params,
    'features': list(X_train_tree.columns),
    'encoding': 'integer'
}

metadata_path = models_dir / "best_tree_model_metadata.json"
with open(metadata_path, 'w') as f:
    json.dump(metadata, f, indent=2)

print(f"Best Tree-Based Model: {best_tree_model_name}")
print(f"Test AUC: {best_tree_score:.4f}")
print(f"\nModel saved to: {model_path}")
print(f"Metadata saved to: {metadata_path}")
print(f"\nHyperparameters:")
print(json.dumps(best_tree_params, indent=2))

Best Tree-Based Model: XGBoost
Test AUC: 0.5993

Model saved to: ..\models\best_tree_model.pkl
Metadata saved to: ..\models\best_tree_model_metadata.json

Hyperparameters:
{
  "n_estimators": 200,
  "learning_rate": 0.3,
  "max_depth": 3,
  "subsample": 1.0
}
